# 3장 1강: 교차표와 카이제곱 독립성 검정 이론 — 실습문제

## 실습 목표

- 두 범주형 변수의 교차표를 구성하고 관측빈도를 해석할 수 있다.
- 행 합계와 열 합계를 이용해 기대빈도를 직접 계산할 수 있다.
- 관측빈도와 기대빈도로 카이제곱 통계량과 자유도를 계산할 수 있다.
- 카이제곱 독립성 검정을 수행하고 기대빈도 조건을 확인할 수 있다.
- p-value와 범주별 비율을 함께 사용하여 변수 간 관련성을 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas, NumPy
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `YearBuilt` | 건축연도 |
| `CentralAir` | 중앙 냉방시설 유무 |
| `KitchenQual` | 주방 품질 |
| `PavedDrive` | 진입로 포장 상태 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> `chi2_contingency()`에서는 강의자료와 동일하게 `correction=False`를 사용합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df = pd.read_csv('ames_housing.csv')
df.head()

,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. 교차표와 카이제곱 통계량 직접 계산

### 문제 1-1. 주택 품질 구간과 중앙 냉방시설의 관계

#### 문제 설명

`OverallQual`을 세 구간으로 나눈 뒤 중앙 냉방시설 유무와의 관계를 확인합니다.

- 낮음: 1~5점
- 보통: 6~7점
- 높음: 8~10점

#### 요구사항

1. `pd.cut()`을 이용해 위 기준으로 `QualityGroup`을 만드세요.
2. 행에는 `QualityGroup`, 열에는 `CentralAir`가 오도록 합계 없는 교차표를 만드세요.
3. `margins=True`인 교차표도 별도로 만들어 행·열 합계를 확인하세요.
4. 합계가 없는 교차표를 NumPy 배열 `observed`로 변환하세요.
5. 행 합계, 열 합계, 전체 합계를 계산하세요.
6. `(행 합계 × 열 합계) / 전체 합계`로 기대빈도를 직접 계산하세요.
7. `Σ(관측-기대)²/기대`로 카이제곱 통계량을 직접 계산하세요.
8. `(행 수-1) × (열 수-1)`로 자유도를 계산하세요.
9. 모든 기대빈도가 5 이상인지 확인하세요.
10. `stats.chi2_contingency(..., correction=False)` 결과와 직접 계산한 값을 비교하세요.
11. p-value를 이용해 두 변수가 관련 있는지 판단하세요.

#### 해석 질문

**Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
**Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
**Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
**Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?

#### 제출 결과

- 관측빈도 교차표와 주변합
- 기대빈도
- 수동 계산한 카이제곱 통계량과 자유도
- 기대빈도 조건 확인
- 함수 결과와의 비교
- 독립성 판단
- Q1~Q4 답변

In [9]:
# 필수 1 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('ames_housing.csv')

# 1. OverallQual을 낮음/보통/높음 3구간으로 나눔
# bins의 경계값은 "초과~이하"로 잘림 (예: 5는 낮음에 포함, 6부터 보통)
bins = [0, 5, 7, 10]
labels = ['낮음', '보통', '높음']
df['QualityGroup'] = pd.cut(df['OverallQual'], bins=bins, labels=labels)

# 2. 행=QualityGroup, 열=CentralAir, 합계 없는 교차표
cross_tab = pd.crosstab(df['QualityGroup'], df['CentralAir'])
print(cross_tab)
print("="*30)

# 3. 행·열 합계를 함께 보여주는 교차표 (기대빈도 계산에 필요한 총계 확인용)
cross_tab_margins = pd.crosstab(df['QualityGroup'], df['CentralAir'], margins=True)
print(cross_tab_margins)
print("="*30)

# 4. 합계 없는 교차표를 순수 숫자 배열로 변환
observed = cross_tab.to_numpy()

# 5. 행 합계(각 품질군의 집 수), 열 합계(냉방 유/무별 집 수), 전체 합계
row_totals = observed.sum(axis=1)   # [538, 693, 229]
col_totals = observed.sum(axis=0)   # [95, 1365]
grand_total = observed.sum()        # 1460

# 6. (행 합계 × 열 합계) / 전체 합계
# np.outer는 두 벡터를 곱해 행렬을 만듦 -> 모든 (행,열) 조합에 대해 곱셈을 한번에 처리
expected = np.outer(row_totals, col_totals) / grand_total
print(np.round(expected, 2))
print("="*30)

# 7. Σ(관측-기대)²/기대
chi2_manual = np.sum((observed - expected)**2 / expected)
print(round(chi2_manual, 4))   # 65.0304
print("="*30)

# 8. 자유도 = (행 수-1) × (열 수-1)
dof_manual = (observed.shape[0] - 1) * (observed.shape[1] - 1)
print(dof_manual)   # 2 (품질군 3개, 냉방여부 2개 → (3-1)×(2-1)=2)
print("="*30)

# 9. 카이제곱 검정이 유효하려면 모든 기대빈도가 5 이상이어야 함
print(expected >= 5)
print("모두 5 이상인가?", np.all(expected >= 5))
print("="*30)

# 10. correction=False → Yates 연속성 보정 없이, 우리가 직접 계산한 방식과 동일하게 맞춤
chi2_scipy, p_value, dof_scipy, expected_scipy = stats.chi2_contingency(observed, correction=False)

print(f"카이제곱통계량 - 직접계산: {chi2_manual:.4f} / scipy: {chi2_scipy:.4f}")
print("="*30)
print(f"자유도         - 직접계산: {dof_manual} / scipy: {dof_scipy}")
print("="*30)
print(f"p-value: {p_value:.4e}")

alpha = 0.05
if p_value < alpha:
    print(f"p-value({p_value:.4e}) < 유의수준({alpha}) → 귀무가설 기각")
    print("→ QualityGroup과 CentralAir는 서로 관련이 있다고 볼 수 있음")
else:
    print("→ 관련 있다고 말할 근거 부족")


CentralAir     N    Y
QualityGroup         
낮음            71  467
보통            23  670
높음             1  228
CentralAir     N     Y   All
QualityGroup                
낮음            71   467   538
보통            23   670   693
높음             1   228   229
All           95  1365  1460
[[ 35.01 502.99]
 [ 45.09 647.91]
 [ 14.9  214.1 ]]
65.0304
2
[[ True  True]
 [ True  True]
 [ True  True]]
모두 5 이상인가? True
카이제곱통계량 - 직접계산: 65.0304 / scipy: 65.0304
자유도         - 직접계산: 2 / scipy: 2
p-value: 7.5654e-15
p-value(7.5654e-15) < 유의수준(0.05) → 귀무가설 기각
→ QualityGroup과 CentralAir는 서로 관련이 있다고 볼 수 있음


### 필수 1 답변 작성란

- **Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
<br>   -> 두 범주형 변수의 해당 조합에 실제로 속한 주택의 개수인 관측빈도

- **Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
<br>   -> 두 범주형 변수가 서로 독립이라는 귀무가설이 참일 때 각 칸에 기대되는 빈도

- **Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
<br>   -> 65.03으로 일치한다. 
- **Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?
<br>   -> 독립적이라고 보기 어려움, p-value가 0.05보다 작아 독립이라는 귀무가설을 기각(서로 연관)

---

## 필수 2. 카이제곱 독립성 검정과 비율 해석

### 문제 2-1. 건축연도 구간과 중앙 냉방시설의 관계

#### 문제 설명

건축연도를 다음 세 구간으로 나누고 중앙 냉방시설 설치 여부와 관련이 있는지 확인하세요.

- 1980년 이전
- 1980~1999년
- 2000년 이후

#### 요구사항

1. `pd.cut()`로 `YearBuiltGroup`을 만드세요.
2. `YearBuiltGroup`과 `CentralAir`의 교차표를 만드세요.
3. 다음 가설을 작성하세요.
   - H₀: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
   - H₁: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 기대빈도 중 5 미만인 칸의 개수를 확인하세요.
7. `pd.crosstab(..., normalize="index")`로 건축연도 구간별 냉방시설 비율을 계산하세요.
8. 검정 결과와 행 비율을 함께 이용해 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
**Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
**Q3.** 기대빈도 조건은 충족되나요?  
**Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
**Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?

#### 제출 결과

- 교차표와 가설
- 카이제곱 검정 결과
- 기대빈도 조건
- 행 비율
- 관련성 및 인과관계 해석
- Q1~Q5 답변

In [12]:
# 필수 2 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('ames_housing.csv')

# 1. 건축연도를 3구간으로 나눔 (bins 경계는 "초과~이하"로 잘림)
bins = [1871, 1979, 1999, 2010]
labels = ['1980년 이전', '1980~1999년', '2000년 이후']
df['YearBuiltGroup'] = pd.cut(df['YearBuilt'], bins=bins, labels=labels)

# 2. 교차표 생성
cross_tab = pd.crosstab(df['YearBuiltGroup'], df['CentralAir'])
print(cross_tab)

observed = cross_tab.to_numpy()
chi2_stat, p_value, dof, expected = stats.chi2_contingency(observed)

print(f"카이제곱 통계량: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"자유도: {dof}")
print(pd.DataFrame(expected, index=cross_tab.index, columns=cross_tab.columns).round(2))

under5_count = (expected < 5).sum()
print(f"기대빈도 5 미만인 칸 개수: {under5_count}")


row_ratio = pd.crosstab(df['YearBuiltGroup'], df['CentralAir'], normalize='index')
print((row_ratio * 100).round(2))


CentralAir       N    Y
YearBuiltGroup         
1980년 이전        95  753
1980~1999년       0  224
2000년 이후         0  388
카이제곱 통계량: 73.3330
p-value: 1.1911e-16
자유도: 2
CentralAir          N       Y
YearBuiltGroup               
1980년 이전        55.18  792.82
1980~1999년      14.58  209.42
2000년 이후        25.25  362.75
기대빈도 5 미만인 칸 개수: 0
CentralAir         N      Y
YearBuiltGroup             
1980년 이전        11.2   88.8
1980~1999년       0.0  100.0
2000년 이후         0.0  100.0


### 필수 2 답변 작성란

- **Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
<br> -> 건축연도 구간과 냉방시설 유무라는 두 범주형 변수의 관계확인, 독립성 검정 사용

- **Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
<br> ->  3행 2열 교차표, (3-1) * (2-1) = 2

- **Q3.** 기대빈도 조건은 충족되나요?  
<br> -> 네, 5미만인 기대 빈도가 0칸으로 기대빈도 기준을 만족.

- **Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
<br> ->있습니다. p-value가 0.05보다 작아 두 변수가 독립적이라는 귀무가설을 기각한다.

- **Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?
<br> -> 없습니다. 카이제곱 독립성 검정은 관련성을 확인할 뿐 인과관계를 증명하지 못함

---

## 과제. 주방 품질과 진입로 포장 상태의 관계

### 문제 3-1. 두 범주형 변수 재분류 후 독립성 검정

#### 문제 설명

기대빈도가 너무 작은 범주를 줄이기 위해 주방 품질과 진입로 포장 상태를 다음과 같이 재분류합니다.

- `KitchenGroup`
  - 우수: `Ex`, `Gd`
  - 보통 이하: `TA`, `Fa`
- `DriveGroup`
  - 완전 포장: `Y`
  - 미포장·부분포장: `N`, `P`

#### 요구사항

1. 위 기준으로 `KitchenGroup`과 `DriveGroup`을 만드세요.
2. 두 변수의 교차표를 작성하세요.
3. 두 변수가 독립이라는 귀무가설과 관련이 있다는 대립가설을 작성하세요.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 모든 기대빈도가 5 이상인지 확인하세요.
7. 주방 품질 집단별 진입로 포장 비율을 계산하세요.
8. 검정 결과와 비율 차이를 함께 사용하여 두 변수의 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요?  
**Q2.** 기대빈도 조건은 충족되나요?  
**Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
**Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?

#### 제출 결과

- 재분류 코드와 교차표
- 가설 설정
- 카이제곱 검정 결과
- 기대빈도 조건 확인
- 행 비율과 결과 해석
- Q1~Q4 답변

In [ ]:
# 과제 코드를 작성하세요.

### 과제 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

---

## 실습 마무리

1. 교차표에서 관측빈도와 기대빈도는 어떻게 다른가요?
<br> -> 관측빈도는 데이터에서 실제로 센 개수, 기대빈도는 두 변수가 독립이라고 가정할 때 주변합으로 예상되는 개수

2. 기대빈도는 어떤 공식으로 계산하나요?
<br> -> (해당 행 x 해당 열) % 전체 합계

3. 카이제곱 통계량이 커진다는 것은 무엇을 의미하나요?
<br> -> 관측빈도가 독립을 가정한 기대빈도에서 전체적으로 더 크게 벗어난다.

4. 독립성 검정과 적합도 검정은 변수 개수와 질문에서 어떻게 다른가요?
<br> -> 독립성 검정은 두 범주형 변수의 관계를 확인하고,
<br> -> 적합도 검정은 하나의 범주형 변수의 관측 분포가 기대한 분포와 일치하는 지 확인한다.

5. 기대빈도가 5보다 작은 칸이 있다면 무엇을 고려해야 하나요?
<br> -> 기대빈도가 작다면 카이제곱 분포로 근사한 p-value의 정확성이 떨어 질 수 있다.